In [1]:
import pandas as pd
df = pd.read_csv("Hydrogen_Uncertainty_dataset.csv")

In [2]:
df

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%),CO2 Captured (tons/hr),Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)
0,7.975000,96.3675,76.5375,83.798065,1.184908
1,10.424999,95.7825,73.5625,83.289360,1.412497
2,7.825000,97.4025,92.6375,84.698059,1.118635
3,3.325000,94.2075,61.4875,81.919785,0.850518
4,8.525001,90.3375,84.0625,78.554565,1.198089
...,...,...,...,...,...
195,2.425000,96.4575,84.5875,83.876320,0.666642
196,7.675000,94.7475,87.5625,82.389351,1.117000
197,4.275000,98.7075,87.0375,85.832848,0.828909
198,2.675000,94.4775,66.9125,82.154579,0.761790


In [3]:
df.isna().sum()

Natural Gas Cost ($/MMBtu, HHV)                                   0
Absorber CO2 Removal Efficiency (%)                               0
Capacity Factor (%)                                               0
CO2 Captured (tons/hr)                                            0
Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
x = df[['Natural Gas Cost ($/MMBtu, HHV)',
    'Absorber CO2 Removal Efficiency (%)',
    'Capacity Factor (%)']]

y = df['Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2)']

In [6]:
x

,"Natural Gas Cost ($/MMBtu, HHV)",Absorber CO2 Removal Efficiency (%),Capacity Factor (%)
0,7.975000,96.3675,76.5375
1,10.424999,95.7825,73.5625
2,7.825000,97.4025,92.6375
3,3.325000,94.2075,61.4875
4,8.525001,90.3375,84.0625
...,...,...,...
195,2.425000,96.4575,84.5875
196,7.675000,94.7475,87.5625
197,4.275000,98.7075,87.0375
198,2.675000,94.4775,66.9125


In [7]:
y

0      1.184908
1      1.412497
2      1.118635
3      0.850518
4      1.198089
         ...   
195    0.666642
196    1.117000
197    0.828909
198    0.761790
199    0.794172
Name: Hydrogen Production Cost: Total Cost: Levelized Cost ($/lb H2), Length: 200, dtype: float64

In [8]:
from sklearn.model_selection import GridSearchCV, train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("\n=== PYOMO MODEL ===\n")


=== PYOMO MODEL ===



In [9]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import Ridge

poly_pipe = Pipeline([
    ('poly',   PolynomialFeatures(include_bias=False)),
    ('scaler', StandardScaler()),
    ('reg',    Ridge())
])
poly_grid = GridSearchCV(
    poly_pipe,
    {'poly__degree': [2, 3, 4],
     'reg__alpha':  [0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5, scoring='r2', n_jobs=-1
)
poly_grid.fit(x_train, y_train)

best_poly = poly_grid.best_estimator_
print("Polynomial best params:", poly_grid.best_params_)
print('Train R2:', best_poly.score(x_train, y_train))
print('Test R2:', best_poly.score(x_test, y_test))

Polynomial best params: {'poly__degree': 4, 'reg__alpha': 0.001}
Train R2: 0.9999880910801492
Test R2: 0.9999768173072929


In [10]:
#Checking the RMSE and MAPE scores
from sklearn.metrics import mean_squared_error
import numpy as np

print("\n=== RMSE & MAPE ===\n")

models = {
    'Polynomial':        best_poly
}

for name, model in models.items():
    y_pred = model.predict(x_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mape   = np.mean(np.abs((y_test - y_pred) / y_test)) * 100
    print(f"{name} | RMSE: {rmse:.4f} | MAPE: {mape:.4f}%")


=== RMSE & MAPE ===

Polynomial | RMSE: 0.0013 | MAPE: 0.1159%


In [11]:
import numpy as np, json, joblib

joblib.dump(best_poly, "Hydrogen_Levelised_Cost_poly.pkl")

poly   = best_poly.named_steps['poly']
scaler = best_poly.named_steps['scaler']
reg    = best_poly.named_steps['reg']

coef_real      = reg.coef_ / scaler.scale_
intercept_real = reg.intercept_ - np.sum(reg.coef_ * scaler.mean_ / scaler.scale_)

feature_names = list(x.columns)
powers = poly.powers_
terms = [{"coef": float(c),
          "powers": {feature_names[j]: int(p[j]) for j in range(len(feature_names)) if p[j] > 0}}
         for p, c in zip(powers, coef_real)]
equation = {"intercept": float(intercept_real), "degree": int(poly.degree), "terms": terms}

P = poly.transform(x_test)
assert np.allclose(intercept_real + P @ coef_real, best_poly.predict(x_test)), "mismatch!"

json.dump(equation, open("Hydrogen_Levelised_Cost_poly_eq.json", "w"), indent=2)
print("saved. degree =", equation["degree"], "| terms =", len(terms))

saved. degree = 4 | terms = 34
